In [1]:
import sys
from pathlib import Path

ML_ROOT = Path.cwd().parent

if str(ML_ROOT) not in sys.path:
    sys.path.append(str(ML_ROOT))

print(ML_ROOT)

import numpy as np
import matplotlib.pyplot as plt
import wfdb
from collections import Counter

from src.datasets.heartbeat_extractor import extract_heartbeat
from src.datasets.record_processor import process_record

c:\Abhiram\Projects\ecg-cardiac-abnormality-detection\ml


In [2]:
record = wfdb.rdrecord("../data/raw/mitdb/100")
annotation = wfdb.rdann("../data/raw/mitdb/100", "atr")

signal = record.p_signal[:, 0]

In [3]:
from src.datasets.dataset_builder import build_dataset
from src.config.constants import MITBIH_RECORDS

X, y, patient_ids = build_dataset(MITBIH_RECORDS)

print(X.shape)
print(y.shape)
print(patient_ids.shape)
print(np.unique(patient_ids))

Processing record 100...
Processing record 101...
Processing record 102...
Processing record 103...
Processing record 104...
Processing record 105...
Processing record 106...
Processing record 107...
Processing record 108...
Processing record 109...
Processing record 111...
Processing record 112...
Processing record 113...
Processing record 114...
Processing record 115...
Processing record 116...
Processing record 117...
Processing record 118...
Processing record 119...
Processing record 121...
Processing record 122...
Processing record 123...
Processing record 124...
Processing record 200...
Processing record 201...
Processing record 202...
Processing record 203...
Processing record 205...
Processing record 207...
Processing record 208...
Processing record 209...
Processing record 210...
Processing record 212...
Processing record 213...
Processing record 214...
Processing record 215...
Processing record 217...
Processing record 219...
Processing record 220...
Processing record 221...


In [4]:
unique_labels, counts = np.unique(y, return_counts=True)

for label, count in zip(unique_labels, counts):
    print(label, count)

A 2546
L 8072
N 75020
R 7255
V 7129


In [5]:
import pandas as pd

df = pd.DataFrame({
    "patient_id": patient_ids,
    "label": y
})

patient_distribution = (
    df.groupby(["patient_id", "label"])
      .size()
      .unstack(fill_value=0)
)

patient_distribution

label,A,L,N,R,V
patient_id,,,,,
100,33,0,2237,0,1
101,3,0,1859,0,0
102,0,0,99,0,4
103,2,0,2081,0,0
104,0,0,163,0,2
105,0,0,2526,0,41
106,0,0,1507,0,520
107,0,0,0,0,59
108,4,0,1738,0,17


In [6]:
import wfdb
import pandas as pd

record_path = "../data/raw/mitdb/102"

annotation = wfdb.rdann(record_path, "atr")

pd.Series(annotation.symbol).value_counts()

/    2028
N      99
f      56
+       5
V       4
Name: count, dtype: int64

In [7]:
from src.preprocessing.splitter import get_patient_splits

train_patients, val_patients, test_patients = get_patient_splits()

print("Train:", train_patients)
print("Validation:", val_patients)
print("Test:", test_patients)

print()

print("Train patients:", len(train_patients))
print("Validation patients:", len(val_patients))
print("Test patients:", len(test_patients))

Train: [100 101 103 105 106 108 109 112 114 115 116 118 121 122 123 200 203 205
 208 209 210 212 213 214 215 219 221 230 232 233 234]
Validation: [117 119 201 202 207 223 228 231]
Test: [102 104 107 111 113 124 217 220 222]

Train patients: 31
Validation patients: 8
Test patients: 9


In [8]:
train_mask = np.isin(patient_ids, train_patients)
val_mask = np.isin(patient_ids, val_patients)
test_mask = np.isin(patient_ids, test_patients)

print("Train")
print(pd.Series(y[train_mask]).value_counts())
print()

print("Validation")
print(pd.Series(y[val_mask]).value_counts())
print()

print("Test")
print(pd.Series(y[test_mask]).value_counts())

Train
N    57922
V     5251
L     4492
R     4387
A     1992
Name: count, dtype: int64

Validation
N    10792
V     1603
L     1457
R     1338
A      250
Name: count, dtype: int64

Test
N    6306
L    2123
R    1530
A     304
V     275
Name: count, dtype: int64
